In [ ]:
import pickle
import pandas as pd
from datetime import timedelta

In [ ]:
def loadmetadata(cxr_data_dir, ehr_data_dir):

    data_dir = cxr_data_dir
    cxr_metadata = pd.read_csv(f'{cxr_data_dir}/mimic-cxr-2.0.0-metadata.csv')
    icu_stay_metadata = pd.read_csv(f'{ehr_data_dir}/root/all_stays.csv')
    columns = ['subject_id', 'stay_id', 'intime', 'outtime']

    # only common subjects with both icu stay and an xray
    cxr_merged_icustays = cxr_metadata.merge(icu_stay_metadata[columns ], how='inner', on='subject_id')

    # combine study date time
    cxr_merged_icustays['StudyTime'] = cxr_merged_icustays['StudyTime'].apply(lambda x: f'{int(float(x)):06}' )
    cxr_merged_icustays['StudyDateTime'] = pd.to_datetime(cxr_merged_icustays['StudyDate'].astype(str) + ' ' + cxr_merged_icustays['StudyTime'].astype(str) ,format="%Y%m%d %H%M%S")

    cxr_merged_icustays.intime=pd.to_datetime(cxr_merged_icustays.intime)
    cxr_merged_icustays.outtime=pd.to_datetime(cxr_merged_icustays.outtime)
    
    end_time = cxr_merged_icustays.intime + pd.DateOffset(hours=48)

    cxr_merged_icustays_during = cxr_merged_icustays.loc[(cxr_merged_icustays.StudyDateTime>=cxr_merged_icustays.intime)&((cxr_merged_icustays.StudyDateTime<=end_time))]
    cxr_merged_icustays_AP = cxr_merged_icustays_during[cxr_merged_icustays_during['ViewPosition'] == 'AP']

    groups = cxr_merged_icustays_AP.groupby('stay_id')

    groups_selected = []
    for group in groups:
        selected = group[1].sort_values('StudyDateTime').tail(1).reset_index()
        groups_selected.append(selected)
    groups = pd.concat(groups_selected, ignore_index=True)

    groups['cxr_time_offset_in_seconds'] = (groups['StudyDateTime'] - groups['intime']).astype('timedelta64[s]')
    return groups

In [ ]:
cxr_dir = '/hdd2/mimic_cxr_jpg/physionet.org/files/mimic-cxr-jpg/2.0.0'
ehr_data_dir = '../../datasets/mimic4extract/data'

In [ ]:
metadata = loadmetadata(cxr_dir, ehr_data_dir)

In [ ]:
metadata

In [ ]:
splits_labels_train = pd.read_csv(f'{ehr_data_dir}/phenotyping/train_listfile.csv')
splits_labels_val = pd.read_csv(f'{ehr_data_dir}/phenotyping/val_listfile.csv')
splits_labels_test = pd.read_csv(f'{ehr_data_dir}/phenotyping/test_listfile.csv')

In [ ]:
train_meta_with_labels = metadata.merge(splits_labels_train, how='inner', on='stay_id')
val_meta_with_labels = metadata.merge(splits_labels_val, how='inner', on='stay_id')
test_meta_with_labels = metadata.merge(splits_labels_test, how='inner', on='stay_id')

In [ ]:
with open('./metas_with_labels_phenotyping_first_48h.pkl', 'wb') as f:
    pickle.dump({
        'train': train_meta_with_labels,
        'val': val_meta_with_labels,
        'test': test_meta_with_labels
    }, f)